In [ ]:
# Round 25 trace3: parse trace2 (k3c+pf+census, 30.10) as reference — per-gap idle table
# and device-busy numbers BEFORE running the new gv2 trace, so the comparison is like-for-like.
import gzip, bisect, resource, sys
from collections import Counter
resource.setrlimit(resource.RLIMIT_AS, (6 << 30, 6 << 30))
import ijson

TRACE2 = 'results/2026-09-21-trace2/traces/dp0_pp0_tp0_dcp0_ep0_rank0.1790009443854305615.pt.trace.json.gz'

dev_events, cpu_by_tid = [], {}
with gzip.open(TRACE2, 'rb') as fh:
    for e in ijson.items(fh, 'traceEvents.item'):
        cat = e.get('cat')
        if cat in ('kernel', 'gpu_memcpy', 'gpu_memset'):
            ts = float(e.get('ts', 0))
            dev_events.append((ts, ts + float(e.get('dur', 0))))
        elif cat in ('cuda_runtime', 'cuda_driver', 'python_function', 'cpu_op'):
            tid = e.get('tid'); ts = float(e.get('ts', 0)); dur = float(e.get('dur', 0))
            if dur <= 300_000.0:
                cpu_by_tid.setdefault(tid, []).append((ts, ts + dur, (e.get('name') or '?')[:150]))

dev_events.sort()
merged = []
for s, e in dev_events:
    if merged and s <= merged[-1][1]:
        merged[-1][1] = max(merged[-1][1], e)
    else:
        merged.append([s, e])

total_busy = sum(e - s for s, e in merged)
total_span = merged[-1][1] - merged[0][0]
print(f'trace2 span {total_span/1e3:.0f} ms, device busy {total_busy/1e3:.0f} ms ({100*total_busy/total_span:.1f}%)')

# All gaps >= 0.5 ms bucketed
gaps = [(e1, s2, s2 - e1) for (_, e1), (s2, _) in zip(merged, merged[1:]) if s2 - e1 >= 500.0]
print(f'trace2 gaps>=0.5ms: {len(gaps)} gaps, total {sum(g for _,_,g in gaps)/1e3:.1f} ms')
ge1 = [g for g in gaps if g >= 1000.0]
print(f'  >=1ms: {len(ge1)} gaps, {sum(ge1)/1e3:.1f} ms')
frag = [g for g in gaps if g < 1000.0]
print(f'  0.5-1ms fragments: {len(frag)} gaps, {sum(frag)/1e3:.1f} ms')
print()
# NOTE: trace2 = k3c+pf+census (PRE-gv2); per-step divisor differs from gv2 trace.


In [ ]:
# Per-gap size histogram for the >=0.5ms pool (trace2 reference)
import numpy as np
gs = np.array([g for _,_,g in gaps]) / 1000.0
hist = Counter()
for g in gs:
    if g >= 10: hist['>=10ms'] += 1
    elif g >= 5: hist['5-10ms'] += 1
    elif g >= 2: hist['2-5ms'] += 1
    elif g >= 1: hist['1-2ms'] += 1
    else: hist['0.5-1ms'] += 1
for k in ['>=10ms','5-10ms','2-5ms','1-2ms','0.5-1ms']:
    n = hist.get(k, 0)
    tot = sum(v for v in gs if (k=='>=10ms' and v>=10) or (k=='5-10ms' and 5<=v<10) or (k=='2-5ms' and 2<=v<5) or (k=='1-2ms' and 1<=v<2) or (k=='0.5-1ms' and v<1))
    print(f'  {k:8s}: {n:4d} gaps, {tot:8.1f} ms')
# number of steps estimate for per-step: Round-22 window 238 steps
STEPS2 = 238
print(f'\nper-step (÷{STEPS2}): total {sum(gs)/STEPS2:.2f} ms, >=1ms {sum(ge1)/1000/STEPS2:.2f} ms, frag {sum(frag)/1000/STEPS2:.2f} ms')
print(f'trace2 device busy per step: {total_busy/1e3/STEPS2:.2f} ms')
